# 01 — Dataset Recreation and Cleaning

Objective: recreate a class-wise plate dataset from scattered raw folders, then enforce cleaning and split contracts used in detector/OCR training.

In [ ]:
import os
import random
import re
import shutil
from pathlib import Path

from PIL import Image
from tqdm import tqdm

In [ ]:
RAW_ROOT = Path('C:/path/to/private/data/raw')
DATASET_DIR = Path('C:/path/to/private/data/DATASET')
FINAL_DIR = Path('C:/path/to/private/data/FINAL_DATASET')

for p in [DATASET_DIR, FINAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('RAW_ROOT   :', RAW_ROOT)
print('DATASET_DIR:', DATASET_DIR)
print('FINAL_DIR  :', FINAL_DIR)

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
PLATE_REGEX = re.compile(r'([A-Z]{2})\s?(\d{1,2})\s?([A-Z]{1,3})\s?(\d{1,4})', re.IGNORECASE)
VALID_PLATE = re.compile(r'^[A-Z]{2}\d{1,2}[A-Z]{1,3}\d{1,4}$')
BLOCKED_KEYWORDS = ['rc', 'book', 'document', 'aadhaar', 'insurance', 'page', 'screenshot', 'tag']

def extract_plate(folder_name: str):
    name = folder_name.replace('(P)', '').strip().upper()
    m = PLATE_REGEX.search(name)
    if not m:
        return None
    return f'{m.group(1)}{m.group(2)}{m.group(3)}{m.group(4)}'

def is_document_like(image_path: Path) -> bool:
    try:
        img = Image.open(image_path)
        w, h = img.size
        ratio = w / max(h, 1)
        if ratio < 0.55 or ratio > 2.2:
            return True
        name = image_path.name.lower()
        return any(k in name for k in BLOCKED_KEYWORDS)
    except Exception:
        return True

In [ ]:
# Step A: recreate class folders from raw structure
copied_images = 0
failed_folders = 0

for root, _, files in tqdm(os.walk(RAW_ROOT), desc='Index raw folders'):
    images = [f for f in files if Path(f).suffix.lower() in IMAGE_EXTENSIONS]
    if not images:
        continue

    plate = extract_plate(Path(root).name)
    if plate is None:
        failed_folders += 1
        continue

    dst_dir = DATASET_DIR / plate
    dst_dir.mkdir(parents=True, exist_ok=True)

    for file in images:
        src = Path(root) / file
        dst = dst_dir / f'{plate}_{Path(file).stem}{Path(file).suffix.lower()}'
        try:
            shutil.copy2(src, dst)
            copied_images += 1
        except Exception:
            pass

print('copied_images =', copied_images)
print('failed_folders =', failed_folders)

In [ ]:
# Step B: cleaning + stratified split contract (70/10/20 at class level)
for split in ['train', 'val', 'test']:
    (FINAL_DIR / split).mkdir(parents=True, exist_ok=True)

removed_invalid = 0
removed_small = 0
removed_docs = 0

for class_dir in tqdm([p for p in DATASET_DIR.iterdir() if p.is_dir()], desc='Clean classes'):
    cls = class_dir.name.upper()
    if not VALID_PLATE.match(cls):
        removed_invalid += 1
        continue

    images = [p for p in class_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS]
    clean_images = []
    for img in images:
        if is_document_like(img):
            removed_docs += 1
            continue
        clean_images.append(img)

    if len(clean_images) <= 6:
        removed_small += 1
        continue

    random.shuffle(clean_images)
    n = len(clean_images)
    n_train = int(0.70 * n)
    n_val = int(0.10 * n)

    split_map = {
        'train': clean_images[:n_train],
        'val': clean_images[n_train:n_train + n_val],
        'test': clean_images[n_train + n_val:]
    }

    for split, imgs in split_map.items():
        dst_cls = FINAL_DIR / split / cls
        dst_cls.mkdir(parents=True, exist_ok=True)
        for img in imgs:
            shutil.copy2(img, dst_cls / img.name)

print('removed_invalid =', removed_invalid)
print('removed_small   =', removed_small)
print('removed_docs    =', removed_docs)

## Deliverable
- `DATASET_DIR`: reconstructed class-wise raw dataset
- `FINAL_DIR/{train,val,test}`: cleaned split used by training notebooks